# Yearly Trends in ARCHER2 Node-Hour Usage

This notebook looks at the yearly changes in workload across the top software applications used on ARCHER2.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

data_path = Path("../data/archer2_HPC_data_without_overall_and_aOut.csv")
hpc_data = pd.read_csv(data_path)

# Make sure the month column is clean and usable
hpc_data["Year"] = pd.to_numeric(hpc_data["Year"], errors="coerce")
hpc_data["Month"] = pd.to_numeric(hpc_data["Month"], errors="coerce")
hpc_data["Nodeh"] = pd.to_numeric(hpc_data["Nodeh"], errors="coerce").fillna(0)

print("Dataset loaded:", hpc_data.shape)
print(hpc_data.head())

In [ ]:
# Identify the top 5 software by total node-hours across all years
top_software = (
    hpc_data.groupby("Software")["Nodeh"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

print("Top 5 software by node-hours:")
print(top_software)

top_names = top_software.index.tolist()
filtered = hpc_data[hpc_data["Software"].isin(top_names)]

In [ ]:
# Aggregate yearly totals for each software
yearly_trend = (
    filtered.groupby(["Year", "Software"])["Nodeh"]
    .sum()
    .reset_index()
)

pivot_df = yearly_trend.pivot_table(
    index="Year",
    columns="Software",
    values="Nodeh",
    aggfunc="sum"
).fillna(0)

print(pivot_df)

In [ ]:
# Plot the yearly trend for each software
pivot_df.plot(
    kind="line",
    figsize=(12, 8),
    linewidth=2,
    marker="*",
    cmap="tab10",
    markersize=7
)

plt.title("Yearly Node-Hour Trend for the Top 5 Software (2021–2026)")
plt.xlabel("Year")
plt.ylabel("Node-hours")
plt.grid(axis="y", linestyle="-", alpha=0.6)
plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()